# Validation: NRTL activity coefficients vs thermo

Compare chemthermo NRTL activity coefficients against `thermo`'s NRTL helper for a binary
mixture. Results are printed as compact tables and a final assertions cell enforces the same
tolerances as `tests/validation/test_flash_vs_thermo.py`.


In [ ]:
from __future__ import annotations

from typing import Any, Iterable

import numpy as np
import chemthermo as ct

try:
    import thermo
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "This notebook requires the 'thermo' package. Install with: pip install thermo"
    ) from exc

NRTL_gammas = thermo.nrtl.NRTL_gammas

# Tolerances aligned with tests/validation/test_flash_vs_thermo.py
GAMMA_TOL = {"rel": 2e-3, "abs": 2e-3}


def _fmt(value: Any) -> str:
    if value is None:
        return "-"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6g}"
    if isinstance(value, (list, tuple, np.ndarray)):
        return "[" + ", ".join(_fmt(float(v)) for v in value) + "]"
    return str(value)


def print_table(
    rows: Iterable[dict[str, Any]],
    columns: list[str] | None = None,
    title: str | None = None,
) -> None:
    rows = list(rows)
    if not rows:
        return
    if columns is None:
        columns = list(rows[0].keys())
    if title:
        print(title)
    formatted = [[_fmt(row.get(col)) for col in columns] for row in rows]
    widths = [max(len(col), max(len(row[i]) for row in formatted)) for i, col in enumerate(columns)]
    header = "  ".join(col.ljust(widths[i]) for i, col in enumerate(columns))
    print(header)
    print("  ".join("-" * widths[i] for i in range(len(columns))))
    for row in formatted:
        print("  ".join(row[i].ljust(widths[i]) for i in range(len(columns))))
    print()


In [ ]:
case = {
    "name": "methane_ethane_240K",
    "components": ["Methane", "Ethane"],
    "zs": [0.50, 0.50],
    "temperature_K": 240.0,
}


In [ ]:
components = case["components"]
zs = case["zs"]
temperature_K = case["temperature_K"]

mixture = ct.Mixture.from_database(components, zs, normalize=True)
model = ct.NRTL(parameters=ct.ActivityParameters.load("NRTL"))
gammas = model.activity_coefficients(
    mixture=mixture,
    temperature_K=temperature_K,
    composition=mixture.fractions,
)

params = ct.NRTLParameters.load()
tau, alpha = params.for_components(components)
ref_gammas = NRTL_gammas(xs=list(mixture.fractions), taus=tau, alphas=alpha)

print_table(
    [
        {
            "case": case["name"],
            "components": components,
            "T [K]": temperature_K,
            "z": zs,
        }
    ],
    title="Input summary",
)

print_table(
    [
        {"source": "chemthermo", "gamma": gammas},
        {"source": "thermo", "gamma": ref_gammas},
    ],
    title="Results",
)

rows = []
for comp, g, rg in zip(components, gammas, ref_gammas):
    diff = float(g) - float(rg)
    rel = diff / (abs(float(rg)) if abs(float(rg)) > 1e-12 else 1.0)
    rows.append({"component": comp, "chem_gamma": g, "thermo_gamma": rg, "abs_diff": diff, "rel_diff": rel})
print_table(rows, title="Differences")


In [ ]:
# Assertions (mirrors tests/validation/test_flash_vs_thermo.py)
assert np.allclose(gammas, ref_gammas, rtol=GAMMA_TOL["rel"], atol=GAMMA_TOL["abs"])
